In [1]:
# Data Quality Checks

# Load a real world dataset
# Run common data quality checks
# Build a simple data quality checklist

In [2]:
%pip install -q pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [3]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)

In [4]:
# Load the dataset
df = pd.read_csv("titanic_dataset.csv")

In [5]:
df.shape

(891, 15)

In [6]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [7]:
# Adding few columns for demonstrating data issues
# ID like column
df["passenger_id"] = np.arange(1, len(df) + 1)

# Constant column
df["constant_col"] = 1

# Create a dirty version of embark_town
df["embark_town_dirty"] = df["embark_town"].copy()

# Select a few random rows (only non missing)
sample_idx = df["embark_town_dirty"].dropna().sample(6, random_state=42).index

# Apply simple dirty transformations
df.loc[sample_idx[0:2], "embark_town_dirty"] = df.loc[sample_idx[0:2], "embark_town_dirty"].str.upper()
df.loc[sample_idx[2:4], "embark_town_dirty"] = df.loc[sample_idx[2:4], "embark_town_dirty"].str.strip().apply(lambda x: f" {x} ")
df.loc[sample_idx[4:6], "embark_town_dirty"] = "unknown"

df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,passenger_id,constant_col,embark_town_dirty
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False,1,1,Southampton
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False,2,1,Cherbourg
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True,3,1,Southampton
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False,4,1,Southampton
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True,5,1,Southampton


In [8]:
# 1. Basic dataset overview
# 2. Missing values summary
# 3. Duplicates
# 4. Data type validation
# 5. Constant and quasi constant columns
# 6. ID like columns
# 7. String inconsistencies
# 8. High null columns
# 9. High zero columns (for numeric features)

In [9]:
# Basic Dataset Overview
print(df.shape)
print("-"*50)
print(df.dtypes)
print("-"*50)
print(df.info())

(891, 18)
--------------------------------------------------
survived               int64
pclass                 int64
sex                      str
age                  float64
sibsp                  int64
parch                  int64
fare                 float64
embarked                 str
class                    str
who                      str
adult_male              bool
deck                     str
embark_town              str
alive                    str
alone                   bool
passenger_id           int64
constant_col           int64
embark_town_dirty        str
dtype: object
--------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   survived           891 non-null    int64  
 1   pclass             891 non-null    int64  
 2   sex                891 non-null    str    
 3   age          

In [10]:
df.columns

Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town',
       'alive', 'alone', 'passenger_id', 'constant_col', 'embark_town_dirty'],
      dtype='str')

In [11]:
df.nunique()

survived               2
pclass                 3
sex                    2
age                   88
sibsp                  7
parch                  7
fare                 248
embarked               3
class                  3
who                    3
adult_male             2
deck                   7
embark_town            3
alive                  2
alone                  2
passenger_id         891
constant_col           1
embark_town_dirty      7
dtype: int64

In [12]:
df.describe()

,survived,pclass,age,sibsp,parch,fare,passenger_id,constant_col
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000,891.000000,891.0
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208,446.000000,1.0
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429,257.353842,0.0
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000,1.000000,1.0
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400,223.500000,1.0
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200,446.000000,1.0
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000,668.500000,1.0
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200,891.000000,1.0


In [ ]:
# Missing values analysis

missing_count = df.isna().sum().sort_values(ascending=False)
missing_percent = (df.isna().mean() * 100).sort_values(ascending=False)

missing_summary = pd.DataFrame(
    {
        "missing_count": missing_count,
        "missing_percent": missing_percent
    }
)
print("Missing Values Summary")
missing_summary

Missing Values Summary


,missing_count,missing_percent
deck,688,77.216611
age,177,19.865320
embark_town_dirty,2,0.224467
embarked,2,0.224467
embark_town,2,0.224467
survived,0,0.000000
pclass,0,0.000000
parch,0,0.000000
sibsp,0,0.000000
sex,0,0.000000


In [14]:
# Duplicates
duplicates_mask = df.duplicated()
num_duplicates = duplicates_mask.sum()

print("Number of duplicate rows:", num_duplicates)

# if you want to remove duplicates:
# df_no_duplicates = df.drop_duplicates()

Number of duplicate rows: 0


In [15]:
# Data Type Validation
expected_types = {
    "survived": "int64",
    "pclass": "int64",
    "sex": "category",
    "age": "float64",
    "fare": "float64",
    "embark_town": "category",
    "passenger_id": "int64"
}

print("Data Type Validation:")
for col, expected in expected_types.items():
    if col in df.columns:
        actual=df[col].dtype
        print(f"{col}: actual={actual}, expected={expected}")

Data Type Validation:
survived: actual=int64, expected=int64
pclass: actual=int64, expected=int64
sex: actual=str, expected=category
age: actual=float64, expected=float64
fare: actual=float64, expected=float64
embark_town: actual=str, expected=category
passenger_id: actual=int64, expected=int64


In [17]:
# example of fixing a data type if needed
# df["pclass"] = df["pclass"].astype("int64")

In [19]:
# Constant & Quasi-Constant columns
nunique = df.nunique()

constant_cols = nunique[nunique == 1].index.to_list()
print("Constant columns:", constant_cols)


# quasi constant columns
quasi_constant_cols = []

for col in df.columns:
    top_freq = df[col].value_counts(normalize=True, dropna=False).values[0]
    if top_freq > 0.95 and col not in constant_cols:
        quasi_constant_cols.append(col)

print("Quasi constant columns (top value more than 95 percent:)", quasi_constant_cols)

Constant columns: ['constant_col']
Quasi constant columns (top value more than 95 percent:) []


In [ ]:
# ID like columns
# Columns where number of unique values is closer to number of df rows
n_rows = len(df)

id_like_cols = []

for col in df.columns:
    if df[col].nunique(dropna=False) == n_rows:
        id_like_cols.append(col)

print("ID like columns:", id_like_cols)

ID like columns: ['passenger_id']


In [21]:
# String Inconsistencies
object_cols = df.select_dtypes(include=["object", "category"]).columns.to_list()
print("Object or category based columns", object_cols)

# simple clean: strip spaces convert it to lower case
df["embark_town_clean"] = (
    df["embark_town_dirty"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace("unknown", np.nan)
)

Object or category based columns ['sex', 'embarked', 'class', 'who', 'deck', 'embark_town', 'alive', 'embark_town_dirty', 'embark_town_clean']


C:\Users\Jatin Dhiman\AppData\Local\Temp\ipykernel_38076\1998947538.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df.select_dtypes(include=["object", "category"]).columns.to_list()


In [1]:
# High null columns
high_null_threshold = 0.4   # 40% or more values are missing in a columns

high_null_cols = missing_summary[missing_summary["missing_percent"] >= high_null_threshold * 100]

print("Columns with high missing percentage:")
print(high_null_cols)

NameError: name 'missing_summary' is not defined

In [ ]:
# High zero columns (numeric)
numeric_cols = df.select_dtypes(include=[np.number]).columns.to_list()
zero_share = {}

for col in numeric_cols:
    zero_share[col] = (df[col] == 0).mean()

zero_share_series = pd.Series(zero_share).sort_values(ascending=False)
print(zero_share_series)

print("-"*50)

high_zero_threshold = 0.8  # 80 percent or more zeros
high_zero_cols = zero_share_series[zero_share_series >= high_zero_threshold]
print("Numeric columns with many zeros:", high_zero_cols)

parch           0.760943
sibsp           0.682379
survived        0.616162
fare            0.016835
pclass          0.000000
age             0.000000
passenger_id    0.000000
constant_col    0.000000
dtype: float64
--------------------------------------------------
Numeric columns with many zeros: Series([], dtype: float64)


In [124]:
df_zeros = df.copy()

df_zeros["col_zeros"] = 0

# numeric_cols = df_zeros.select_dtypes(include=[np.number]).columns.to_list()
numeric_cols = ["age", "constant_col", "fare", "col_zeros"]
zero_share = {}

for col in numeric_cols:
    zero_share[col] = (df_zeros[col] == 0).mean()

zero_share_series = pd.Series(zero_share).sort_values(ascending=False)
print(zero_share_series)

print("-"*50)

high_zero_threshold = 0.8  # 80 percent or more zeros
high_zero_cols = zero_share_series[zero_share_series >= high_zero_threshold]
print("Numeric columns with many zeros:", high_zero_cols)

col_zeros       1.000000
fare            0.016835
age             0.000000
constant_col    0.000000
dtype: float64
--------------------------------------------------
Numeric columns with many zeros: col_zeros    1.0
dtype: float64


In [16]:
# Data Quality Checks

In [125]:
print("Data quality checklist summary")
print("- Shape and columns checked")
print("- Missing values summary created")
print("- Duplicates counted and removable copy created")
print("- Data types compared with expectations")
print("- Constant and quasi constant columns flagged")
print("- ID like columns detected")
print("- String inconsistencies checked and cleaned example")
print("- High null columns identified")
print("- High zero numeric columns identified")

Data quality checklist summary
- Shape and columns checked
- Missing values summary created
- Duplicates counted and removable copy created
- Data types compared with expectations
- Constant and quasi constant columns flagged
- ID like columns detected
- String inconsistencies checked and cleaned example
- High null columns identified
- High zero numeric columns identified
